# Tema 6 · Proyecto aplicado de visión artificial

Taller de 150 minutos en parejas. Guarda una copia y completa las respuestas. Elige una de las cuatro rutas del enunciado: clasificación, detección/segmentación, anomalías o búsqueda visual.

Este notebook incluye un **ejemplo ejecutable de clasificación con datos sintéticos**, para que puedas empezar sin descargas ni GPU. Es un punto de partida, no una solución entregable sin cambios. Para otras rutas, reutiliza su notebook del curso y conserva las secciones de encargo, decisiones, evidencias y defensa de esta plantilla. No es obligatorio ejecutar el ejemplo de clasificación si eliges otra ruta.

La prueba del ejemplo permanece cerrada por defecto: `EVALUAR_PRUEBA = False`. Solo actívala cuando hayas cerrado tu selección en validación. El interruptor ayuda a ordenar el trabajo; no constituye un control técnico contra mirar los datos.

## Actividad 1 · Formular el encargo (10 min)

Completa quién usará el resultado, qué decisión tomará y qué salida necesita. El ejemplo clasifica dibujos de tres familias; no valida inspección de piezas reales.

**Hipótesis:** ¿qué cambio propones y por qué esperas que ayude?

**Criterio de éxito:** escríbelo antes de ver prueba. Puede incluir condiciones bajo las cuales decidirías no continuar.

**Tu encargo e hipótesis:** …

In [ ]:
PROYECTO = {
    "equipo": "Completar",
    "ruta": "clasificacion",
    "usuario_y_decision": "Completar",
    "hipotesis": "Completar",
    "criterio_exito": "Completar antes de evaluar prueba",
    "origen_datos": "Ejemplo: dibujos sintéticos generados por código",
    "cambio_realizado": "Completar",
    "limites_y_siguiente_paso": "Completar",
}

## Actividad 2 · Datos y separación (15 min)

**Para cualquier ruta:** identifica el origen, las etiquetas, la unidad independiente y las particiones. No pongas versiones de la misma pieza en conjuntos distintos.

**Ejemplo A:** creamos una nueva realización del generador del tema 2, con otras semillas: 180 piezas sintéticas y tres capturas por pieza. Las formas son arandela, placa y escuadra. No son fotografías. La separación es por pieza. Entrenamiento ajusta pesos; validación selecciona opciones; prueba estima el resultado final.

**Tu descripción de datos, unidad y separación:** …

In [ ]:
from pathlib import Path
import json, platform, hashlib
from importlib.metadata import version
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report
from sklearn.dummy import DummyClassifier

NOMBRES = np.array(["Arandela", "Placa", "Escuadra"])
def dibujar(clase, radio, desplazamiento=(0, 0), luz=1.0, ruido=0.04, seed=0):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[-16:16, -16:16]
    xx = xx - desplazamiento[0]
    yy = yy - desplazamiento[1]
    if clase == 0:
        distancia = np.sqrt(xx**2 + yy**2)
        mascara = (distancia < radio) & (distancia > radio * 0.5)
    elif clase == 1:
        mascara = (np.abs(xx) < radio) & (np.abs(yy) < radio * 0.7)
    else:
        mascara = ((np.abs(xx) < radio) & (np.abs(yy) < radio)
                   & ((xx < -radio * 0.25) | (yy > radio * 0.25)))
    imagen = np.where(mascara, 0.75, 0.15) * luz
    return np.clip(imagen + rng.normal(0, ruido, imagen.shape), 0, 1).astype(np.float32)

In [ ]:
rng = np.random.default_rng(2026)
imagenes, etiquetas, grupos = [], [], []
for clase in range(3):
    for pieza in range(60):
        radio = rng.uniform(7, 12)
        grupo = clase * 60 + pieza
        for vista in range(3):
            imagenes.append(dibujar(clase, radio,
                desplazamiento=tuple(rng.integers(-3, 4, size=2)),
                luz=rng.uniform(0.8, 1.2), seed=6000 + grupo * 3 + vista))
            etiquetas.append(clase)
            grupos.append(grupo)
imagenes = np.stack(imagenes)
y = np.array(etiquetas)
grupos = np.array(grupos)
X = imagenes.reshape(len(imagenes), -1)
print("Imágenes:", imagenes.shape, "Entrada por imagen:", X.shape[1])
fig, axes = plt.subplots(3, 4, figsize=(8, 6))
for clase in range(3):
    ids = np.flatnonzero(y == clase)[::15][:4]
    for ax, idx in zip(axes[clase], ids):
        ax.imshow(imagenes[idx], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{NOMBRES[y[idx]]} · {grupos[idx]}")
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
split1 = GroupShuffleSplit(n_splits=1, test_size=0.4, random_state=41)
train, resto = next(split1.split(X, y, groups=grupos))
split2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=43)
a, b = next(split2.split(X[resto], y[resto], groups=grupos[resto]))
val, test = resto[a], resto[b]
assert not (set(grupos[train]) & set(grupos[val]))
assert not (set(grupos[train]) & set(grupos[test]))
assert not (set(grupos[val]) & set(grupos[test]))
for nombre, idx in [("Entrenamiento", train), ("Validación", val), ("Prueba", test)]:
    print(nombre, len(idx), "imágenes; por clase:", np.bincount(y[idx], minlength=3))
    assert len(np.unique(y[idx])) == 3
particiones = []
for nombre, idx in [("entrenamiento", train), ("validacion", val), ("prueba", test)]:
    for i in idx:
        particiones.append({"imagen_id": int(i), "pieza_id": int(grupos[i]),
                            "clase": str(NOMBRES[y[i]]), "particion": nombre})
particiones = pd.DataFrame(particiones)
print("Piezas por partición:")
print(particiones.groupby("particion")["pieza_id"].nunique().to_string())

## Actividad 3 · Comparar en validación (25 min)

**Cualquier ruta:** define una referencia y una propuesta; cambia una familia de parámetros y registra resultados en validación. Evita buscar muchas opciones sin una hipótesis.

**Ejemplo A:** referencia = predecir siempre la clase más frecuente. Propuesta = regresión logística con escalado ajustado solo en entrenamiento. Comparamos dos valores de regularización. Los candidatos se eligen por acierto en validación; en empate se conserva el primero. Puedes modificar los candidatos aquí antes de abrir prueba.

**Para que sea tu proyecto:** introduce y justifica un cambio, por ejemplo reducir resolución de entrada, usar menos piezas de entrenamiento o variar regularización. Documenta también si no mejora.

**Tu referencia, propuesta y cambio:** …

In [ ]:
CANDIDATOS_C = [0.01, 0.1]
referencia = DummyClassifier(strategy="most_frequent").fit(X[train], y[train])
modelos = []
filas = [{"Modelo": "Mayoría", "C": None,
          "Acierto validación": accuracy_score(y[val], referencia.predict(X[val]))}]
for c in CANDIDATOS_C:
    modelo = make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=500))
    modelo.fit(X[train], y[train])
    modelos.append(modelo)
    filas.append({"Modelo": "Lineal", "C": c,
                  "Acierto validación": accuracy_score(y[val], modelo.predict(X[val]))})
resultados_val = pd.DataFrame(filas)
print(resultados_val.round(3).to_string(index=False))
mejor = int(np.argmax([m.score(X[val], y[val]) for m in modelos]))
modelo_elegido = modelos[mejor]
# La referencia también es una opción: una propuesta peor no se selecciona.
usar_propuesta = modelo_elegido.score(X[val], y[val]) > referencia.score(X[val], y[val])
modelo_final = modelo_elegido if usar_propuesta else referencia
DECISION = {
    "modelo": "Lineal" if usar_propuesta else "Mayoría",
    "C": float(CANDIDATOS_C[mejor]) if usar_propuesta else None,
    "criterio": "Mayor acierto de validación; empate frente a referencia favorece referencia",
    "acierto_validacion": float(modelo_final.score(X[val], y[val])),
}
print("Selección por validación:", DECISION)

### Para las rutas B, C y D

- **Detección/segmentación:** reutiliza tema 3. Compara una máscara sin limpieza con otra filtrada, o dos criterios de área, usando validación. Fija umbral y criterio de IoU antes de prueba. Si evalúas cajas, conserva emparejamiento uno a uno; no lo llames mAP.
- **Anomalías:** reutiliza tema 4. Compara criterios de umbral en validación con referencia normal fija. Informa FP/FN y sus denominadores, además de resultados por tipo de defecto.
- **Búsqueda visual:** reutiliza tema 5. Define antes qué imágenes son relevantes para cada consulta y compara una referencia sencilla (por ejemplo, orden fijo) con CLIP. Cuenta aciertos de la primera posición y documenta casos sin respuesta. Separa consultas de desarrollo de las de evaluación; con este catálogo diminuto, las conclusiones serán exploratorias. La revisión de renders puede complementar el proyecto, pero un render por sí solo no es evaluación de visión.

No es necesario instalar modelos nuevos para ninguna ruta. La plantilla numérica siguiente corresponde solo a la ruta A.

## Actividad 4 · Cerrar decisiones y evaluar (12 min)

Antes de activar la celda, registra qué modelo eliges, por qué y con qué criterio juzgarás el resultado. Activa `EVALUAR_PRUEBA` solo después. Si modificas el sistema tras consultar prueba, debes desarrollar la mejora con entrenamiento/validación y reservar nuevos datos para otra estimación final.

**Tu decisión cerrada y fecha de cierre:** …

In [ ]:
EVALUAR_PRUEBA = False  # Cambia a True solo tras cerrar decisiones.
RESUMEN_PRUEBA = None
pred_prueba = None
if EVALUAR_PRUEBA:
    pred_base = referencia.predict(X[test])
    pred_prueba = modelo_final.predict(X[test])
    RESUMEN_PRUEBA = {
        "n_imagenes": int(len(test)), "n_piezas": int(len(np.unique(grupos[test]))),
        "acierto_referencia": float(accuracy_score(y[test], pred_base)),
        "acierto_modelo_elegido": float(accuracy_score(y[test], pred_prueba)),
        "decision": DECISION,
    }
    print(json.dumps(RESUMEN_PRUEBA, ensure_ascii=False, indent=2))
    print(classification_report(y[test], pred_prueba, labels=np.arange(3),
                                target_names=NOMBRES, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y[test], pred_prueba, labels=np.arange(3),
                                           display_labels=NOMBRES, cmap="Blues")
    plt.title("Prueba · filas reales, columnas predichas")
    plt.tight_layout()
    plt.show()
else:
    print("Prueba sin evaluar. Revisa validación y registra la decisión antes de activarla.")

## Actividad 5 · Interpretar y preparar la entrega (12 min)

Examina dos errores o limitaciones. Si no hay errores en la muestra, no los inventes: explica qué no se ha probado. Una prueba de 108 imágenes de 36 piezas no equivale a 108 piezas independientes. No generalices dibujos sintéticos a fabricación real.

**Caso 1 — observación, causa posible, comprobación futura:** …

**Caso 2 — observación, causa posible, comprobación futura:** …

**Conclusión — qué decisión permite tomar esta evidencia y cuál sigue pendiente:** …

In [ ]:
if pred_prueba is not None:
    fallos = np.flatnonzero(pred_prueba != y[test])
    if len(fallos):
        fig, axes = plt.subplots(1, min(4, len(fallos)), figsize=(10, 3), squeeze=False)
        for ax, pos in zip(axes[0], fallos[:4]):
            ax.imshow(imagenes[test[pos]], cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"Real: {NOMBRES[y[test[pos]]]}\nPred.: {NOMBRES[pred_prueba[pos]]}")
            ax.axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print("Sin errores observados: documenta dos limitaciones del conjunto y la evaluación.")
else:
    print("Todavía no hay predicciones de prueba; prepara la discusión sin inventar resultados.")

### Exportar evidencias del ejemplo A

Esta celda guarda un informe **en borrador**, las particiones y las métricas disponibles en `resultados/tema6/`. No redacta tus conclusiones ni entrega el trabajo por ti. La exportación se guarda en una subcarpeta identificada por el estado de la evaluación para no mezclar resultados pendientes y evaluados. Descarga también tu notebook con salidas; los archivos de Colab son temporales. Para otras rutas, adapta la exportación y los nombres de métricas.

In [ ]:
ESTADO = "evaluado" if RESUMEN_PRUEBA is not None else "pendiente"
SALIDA = Path("resultados") / "tema6" / ESTADO
SALIDA.mkdir(parents=True, exist_ok=True)
resultados_val.to_csv(SALIDA/"validacion.csv", index=False)
particiones.to_csv(SALIDA/"particiones.csv", index=False)
registro = {"estado": ESTADO, "proyecto": PROYECTO, "seleccion": DECISION,
            "prueba": RESUMEN_PRUEBA,
            "versiones": {"python": platform.python_version(),
                          **{p: version(p) for p in ["numpy", "pandas", "scikit-learn", "matplotlib"]}},
            "huella_datos_sinteticos": hashlib.sha256(imagenes.tobytes()+y.tobytes()+grupos.tobytes()).hexdigest()}
(SALIDA/"registro.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
if pred_prueba is not None:
    pd.DataFrame({"imagen_id": test, "pieza_id": grupos[test],
                  "real": y[test], "prediccion": pred_prueba}).to_csv(SALIDA/"predicciones_prueba.csv", index=False)
informe = "# Proyecto de visión · BORRADOR\n\n"
informe += "Estado de prueba: " + ESTADO + "\n\n"
for clave, valor in PROYECTO.items():
    informe += "## " + clave.replace("_", " ").capitalize() + "\n\n" + str(valor) + "\n\n"
informe += "## Evidencias\n\nVer validacion.csv, particiones.csv y registro.json. "
informe += "Añade dos casos analizados y una conclusión propia. Guarda y adjunta el notebook ejecutado.\n"
(SALIDA/"informe_borrador.md").write_text(informe, encoding="utf-8")
print("Evidencias guardadas en", SALIDA.resolve())
print("El informe mantiene la marca BORRADOR: complétalo antes de entregar.")

## Actividad 6 · Defensa y revisión entre equipos (bloque común: 35 min)

Prepara una defensa de **5 minutos**: problema y datos (1 min), método y comparación (1 min), resultados y dos errores/límites (2 min), conclusión y siguiente paso (1 min). No necesitas crear otra presentación: puedes mostrar el notebook.

El bloque permite **hasta cinco equipos**, con 5 min de defensa + 2 min de preguntas por equipo. Si hay más, usa estaciones simultáneas en las que un equipo presenta y otro revisa: dos rondas de 15 min y 5 min para recoger conclusiones. Alternad portavoces; la guía docente explica la adaptación.

**Revisión al otro equipo:** una decisión bien justificada, una limitación que falta y una pregunta sobre los datos. Usa la rúbrica del tema.

## Entrega

- Notebook guardado con código, salidas y respuestas propias.
- Informe de una página aproximadamente: encargo, datos, comparación, evidencias, límites y siguiente paso.
- Archivos de resultados pertinentes, sin datos personales.

Antes de entregar, reinicia el kernel y ejecuta por orden la versión final. Conserva las decisiones cerradas; repetir sin cambios para comprobar ejecución no debe convertirse en ajustar por prueba. Si el ejemplo A sigue marcado como pendiente, no presentes resultados finales de prueba.

La rúbrica valora problema/datos (25 %), método (25 %), evaluación y errores (30 %) y claridad/reproducibilidad (20 %). No se califica solo el porcentaje de acierto. La ponderación de este proyecto en la asignatura corresponde a la coordinación docente.